In [2]:
import os
from google.colab import drive

# 1. Mount Google Drive
drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import os
import re
import io as pyio
import gdown

DATA_DIR = "/content/welding_data_0"
OUTPUT_DIR = "/content/welding_data/extension_0"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

FOLDER_URL = input("Paste the Google Drive folder URL from Chrome: ").strip()

# Extract folder identifier dynamically from the URL
match = re.search(r"/folders/([a-zA-Z0-9_-]+)", FOLDER_URL)
if not match:
    raise ValueError("Invalid Google Drive folder URL format.")
folder_key = match.group(1)

print("Attempting direct folder download via gdown...")
try:
    gdown.download_folder(
        url=FOLDER_URL,
        output=DATA_DIR,
        quiet=False,
        use_cookies=False
    )
except Exception as e:
    print(f"gdown encountered an issue: {e}")

# Verify if the core HDF5 files were downloaded; if not, use authenticated Drive API stream
required_files = [
    "mechanical_elastoplastic.h5",
    "mechanical_elastoplastic.xdmf",
    "thermal.h5"
]
missing_files = [
    f for f in required_files if not os.path.exists(os.path.join(DATA_DIR, f))
]

if missing_files:
    print("Switching to authenticated Drive API stream for large binary files...")
    from google.colab import auth
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload

    auth.authenticate_user()
    drive_service = build("drive", "v3")

    query = f"'{folder_key}' in parents and trashed = false"
    results = drive_service.files().list(
        q=query,
        pageSize=50,
        fields="files(id, name, size)",
        supportsAllDrives=True,
        includeItemsFromAllDrives=True
    ).execute()

    items = results.get("files", [])
    if not items:
        raise RuntimeError(
            "No files found. Ensure the link sharing permission in Chrome is set to 'Anyone with the link'."
        )

    for item in items:
        file_name = item["name"]
        dest_path = os.path.join(DATA_DIR, file_name)
        if os.path.exists(dest_path) and os.path.getsize(dest_path) > 0:
            print(f"Already exists: {file_name}")
            continue

        print(f"Downloading {file_name} ...")
        request = drive_service.files().get_media(fileId=item["id"])
        with open(dest_path, "wb") as fh:
            downloader = MediaIoBaseDownload(fh, request, chunksize=50 * 1024 * 1024)
            done = False
            while not done:
                status, done = downloader.next_chunk()
                if status:
                    print(f"  {file_name}: {int(status.progress() * 100)}%")

print("\nDownloaded files in Colab:", os.listdir(DATA_DIR))

Paste the Google Drive folder URL from Chrome: https://drive.google.com/drive/folders/18R-EeZBpHHTf9Dpv04cxxzvRVQwzE-xS
Attempting direct folder download via gdown...


Retrieving folder contents


Processing file 1UaYWR7Fz_XXL31RZKT1bT28pJIDxgryU final_results.h5
Processing file 1ogvVjcDGfRQWbepQjfm1l8-fcnEZVEcT final_results.xdmf
Processing file 1moWtW_0PmekEIjLkosjQvlnTF6LEjSSX mechanical_elastoplastic.h5
Processing file 1A14Wne_K3vRcuakXAOA3kgLDlhEm6ajQ mechanical_elastoplastic.xdmf
Processing file 1qqy5p3aq-849DwSMuqNOghK4ozl0yKdR mechanical_results.jsonl
Processing file 1cDonJMlrQFJx0SVfO3nG38Sqhsj2yJiu mesh_topology.json
Processing file 1xznCdAyhSf4r7vg4PNl5--BtTieCJQ88 thermal_series.jsonl
Processing file 1Q32cJIQbQaD2oDdMpZVKE4MNLa9zYV1D thermal.h5
Processing file 1TyM0Voevc3HEW18vZIp5UBPWMkL7CwI6 thermal.xdmf
Processing file 1OSBoQMCHuWDYeZqzhBCBCn24wcNInAYP volumetric_elements.jsonl
Processing file 1C32MR0zv5vROCyq3TAi4wV5NmeRPECiU weld_mesh_final.msh


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1UaYWR7Fz_XXL31RZKT1bT28pJIDxgryU
To: /content/welding_data_0/final_results.h5
100%|██████████| 308k/308k [00:00<00:00, 56.2MB/s]
Downloading...
From: https://drive.google.com/uc?id=1ogvVjcDGfRQWbepQjfm1l8-fcnEZVEcT
To: /content/welding_data_0/final_results.xdmf
100%|██████████| 1.10k/1.10k [00:00<00:00, 4.65MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1moWtW_0PmekEIjLkosjQvlnTF6LEjSSX
From (redirected): https://drive.google.com/uc?id=1moWtW_0PmekEIjLkosjQvlnTF6LEjSSX&confirm=t&uuid=e90bb75c-80b5-4ef5-8325-0dab808e1f14
To: /content/welding_data_0/mechanical_elastoplastic.h5
100%|██████████| 142M/142M [00:01<00:00, 110MB/s]
Downloading...
From: https://drive.google.com/uc?id=1A14Wne_K3vRcuakXAOA3kgLDlhEm6ajQ
To: /content/welding_data_0/mechanical_elastoplastic.xdmf
100%|██████████| 158k/158k [00:00<00:00, 27.


Downloaded files in Colab: ['thermal.h5', 'final_results.h5', 'mechanical_elastoplastic.h5', 'mechanical_elastoplastic.xdmf', 'volumetric_elements.jsonl', 'mechanical_results.jsonl', 'thermal_series.jsonl', 'thermal.xdmf', 'mesh_topology.json', 'weld_mesh_final.msh', 'final_results.xdmf']



Download completed


# Test file data

In [6]:
import os
import json
import h5py
import numpy as np
import xml.etree.ElementTree as ET

DATA_DIR = "/content/welding_data_0"
if not os.path.exists(DATA_DIR):
    DATA_DIR = "."

print(f"=== Scanning Directory: {os.path.abspath(DATA_DIR)} ===")
all_files = sorted(os.listdir(DATA_DIR))
for f_name in all_files:
    f_path = os.path.join(DATA_DIR, f_name)
    if os.path.isfile(f_path):
        size_mb = os.path.getsize(f_path) / (1024 * 1024)
        print(f" - {f_name} ({size_mb:.2f} MB)")


def inspect_h5_file(filename):
    filepath = os.path.join(DATA_DIR, filename)
    if not os.path.exists(filepath):
        print(f"\n[SKIP] {filename} not found.")
        return

    print(f"\n==========================================")
    print(f"Inspecting HDF5: {filename}")
    print(f"==========================================")
    with h5py.File(filepath, "r") as f:
        print("Top-level groups:", list(f.keys()))

        # 1. Check Mesh and Topology
        if "Mesh" in f:
            for m_name in f["Mesh"].keys():
                m_grp = f["Mesh"][m_name]
                print(f"\n[Mesh Group: '{m_name}'] -> Subkeys: {list(m_grp.keys())}")
                for subk in m_grp.keys():
                    item = m_grp[subk]
                    if isinstance(item, h5py.Dataset):
                        print(f"  - Dataset '{subk}': shape={item.shape}, dtype={item.dtype}")
                    elif isinstance(item, h5py.Group):
                        for d_name, d_val in item.items():
                            if isinstance(d_val, h5py.Dataset):
                                print(f"  - '{subk}/{d_name}': shape={d_val.shape}, dtype={d_val.dtype}")

        # 2. Check MeshTags / Boundary conditions if stored
        for tag_key in ["MeshTags", "meshtags", "FacetTags", "CellTags"]:
            if tag_key in f:
                print(f"\n[Tags Group: '{tag_key}'] -> Keys: {list(f[tag_key].keys())}")

        # 3. Check Stored Functions / Time Steps
        if "Function" in f:
            print("\n[Function Group (Saved Physical Fields)]:")
            for func_name in f["Function"].keys():
                grp = f["Function"][func_name]
                step_keys = list(grp.keys())
                try:
                    step_keys_sorted = sorted(step_keys, key=lambda k: float(k.replace("_", ".")))
                except ValueError:
                    step_keys_sorted = sorted(step_keys)

                first_k = step_keys_sorted[0]
                last_k = step_keys_sorted[-1]
                last_arr = np.array(grp[last_k])

                print(f"  * Field Name: '{func_name}'")
                print(f"    - Total saved steps: {len(step_keys_sorted)}")
                print(f"    - First step key: '{first_k}' | Last step key: '{last_k}'")
                print(f"    - Array shape at last step: {last_arr.shape}, dtype: {last_arr.dtype}")
                print(f"    - Last step range: min={np.min(last_arr):.6g}, max={np.max(last_arr):.6g}")


def inspect_xdmf_file(filename):
    filepath = os.path.join(DATA_DIR, filename)
    if not os.path.exists(filepath):
        print(f"\n[SKIP] {filename} not found.")
        return

    print(f"\n==========================================")
    print(f"Inspecting XDMF: {filename}")
    print(f"==========================================")
    try:
        tree = ET.parse(filepath)
        root = tree.getroot()
        time_vals = [ float(t.attrib["Value"]) for t in root.iter("Time") if "Value" in t.attrib ]
        attr_names = sorted(list({ a.attrib.get("Name", "Unknown") for a in root.iter("Attribute") }))
        print(f" - Attribute Names in XDMF: {attr_names}")
        if time_vals:
            print(f" - Time entries count: {len(time_vals)} | Min Time: {min(time_vals)} | Max Time: {max(time_vals)}")
    except Exception as e:
        print(f" - Error parsing XDMF: {e}")


def inspect_json_file(filename):
    filepath = os.path.join(DATA_DIR, filename)
    if not os.path.exists(filepath):
        print(f"\n[SKIP] {filename} not found.")
        return

    print(f"\n==========================================")
    print(f"Inspecting JSON: {filename}")
    print(f"==========================================")
    with open(filepath, "r") as f:
        data = json.load(f)
    if isinstance(data, dict):
        print(" - JSON top-level keys:", list(data.keys()))
        for k, v in data.items():
            if isinstance(v, list):
                sample = v[0] if len(v) > 0 else None
                print(f"   * '{k}': list of length {len(v)}, first element sample: {sample}")
            else:
                print(f"   * '{k}': type={type(v).__name__}, value={v}")


def inspect_jsonl_file(filename):
    filepath = os.path.join(DATA_DIR, filename)
    if not os.path.exists(filepath):
        print(f"\n[SKIP] {filename} not found.")
        return

    print(f"\n==========================================")
    print(f"Inspecting JSONL: {filename}")
    print(f"==========================================")
    first_rec = None
    last_rec = None
    count = 0
    with open(filepath, "r") as f:
        for line in f:
            if line.strip():
                count += 1
                if first_rec is None:
                    first_rec = json.loads(line)
                last_rec = json.loads(line)

    print(f" - Total lines (frames): {count}")
    if first_rec and last_rec:
        print(f" - Keys in each frame: {list(last_rec.keys())}")
        print(f" - First frame time: {first_rec.get('time')} | Last frame time: {last_rec.get('time')}")
        for k, v in last_rec.items():
            if isinstance(v, list):
                print(f"   * '{k}': list length={len(v)}")


def inspect_msh_file(filename):
    filepath = os.path.join(DATA_DIR, filename)
    if not os.path.exists(filepath):
        print(f"\n[SKIP] {filename} not found in this folder.")
        return

    print(f"\n==========================================")
    print(f"Inspecting Gmsh MSH: {filename}")
    print(f"==========================================")
    sections = []
    physical_names = []
    with open(filepath, "r", errors="ignore") as f:
        in_phys = False
        for line in f:
            s = line.strip()
            if s.startswith("$") and not s.startswith("$End"):
                sections.append(s)
                if s == "$PhysicalNames":
                    in_phys = True
                continue
            if s == "$EndPhysicalNames":
                in_phys = False
                continue
            if in_phys:
                physical_names.append(s)
    print(" - MSH Sections present:", sections)
    if physical_names:
        print(" - Physical Names / Boundary Groups:", physical_names[1:])
    else:
        print(" - No $PhysicalNames section defined in MSH file.")


# Run inspections on all target files
inspect_h5_file("mechanical_elastoplastic.h5")
inspect_h5_file("thermal.h5")
inspect_xdmf_file("mechanical_elastoplastic.xdmf")
inspect_xdmf_file("final_results.xdmf")
inspect_json_file("mesh_topology.json")
inspect_jsonl_file("thermal_series.jsonl")
inspect_msh_file("weld_mesh_final.msh")

=== Scanning Directory: /content/welding_data_0 ===
 - final_results.h5 (0.29 MB)
 - final_results.xdmf (0.00 MB)
 - mechanical_elastoplastic.h5 (135.40 MB)
 - mechanical_elastoplastic.xdmf (0.15 MB)
 - mechanical_results.jsonl (943.37 MB)
 - mesh_topology.json (1.06 MB)
 - thermal.h5 (114.44 MB)
 - thermal.xdmf (0.42 MB)
 - thermal_series.jsonl (99.13 MB)
 - volumetric_elements.jsonl (1.25 MB)
 - weld_mesh_final.msh (1.57 MB)

Inspecting HDF5: mechanical_elastoplastic.h5
Top-level groups: ['Function', 'Mesh']

[Mesh Group: 'Grid'] -> Subkeys: ['geometry', 'topology']
  - Dataset 'geometry': shape=(16226, 3), dtype=float64
  - Dataset 'topology': shape=(13320, 8), dtype=int64

[Function Group (Saved Physical Fields)]:
  * Field Name: 'Displacement'
    - Total saved steps: 51
    - First step key: '0_02' | Last step key: '100'
    - Array shape at last step: (16226, 3), dtype: float64
    - Last step range: min=-0.000168232, max=0.00131931
  * Field Name: 'EquivalentPlasticStrain'
    

# analysis grid

In [7]:
import os
import json
import h5py
import numpy as np

DATA_DIR = "/content/welding_data_0"

# 1. Check geometry bounding box and exact time step keys in HDF5
with h5py.File(os.path.join(DATA_DIR, "mechanical_elastoplastic.h5"), "r") as f:
    geom = np.array(f["Mesh/Grid/geometry"])
    print("Geometry Bounding Box:")
    print(f"  X: min={geom[:,0].min():.6f}, max={geom[:,0].max():.6f}")
    print(f"  Y: min={geom[:,1].min():.6f}, max={geom[:,1].max():.6f}")
    print(f"  Z: min={geom[:,2].min():.6f}, max={geom[:,2].max():.6f}")

    disp_grp = f["Function/Displacement"]
    keys = sorted(list(disp_grp.keys()), key=lambda k: float(k.replace("_", ".")))
    print(f"\nMechanical HDF5 step keys ({len(keys)} total):")
    print("  First 5:", keys[:5])
    print("  Last 5:", keys[-5:])

    # Find nodes where Displacement is exactly [0, 0, 0] at t=100s (Clamped BC detection)
    u_100 = np.array(disp_grp["100"])
    zero_disp_mask = np.linalg.norm(u_100, axis=1) < 1e-12
    clamped_coords = geom[zero_disp_mask]
    print(f"\nDetected {np.sum(zero_disp_mask)} clamped nodes (u=0 at t=100s):")
    if len(clamped_coords) > 0:
        print(f"  Clamped X range: [{clamped_coords[:,0].min():.6f}, {clamped_coords[:,0].max():.6f}]")
        print(f"  Clamped Y range: [{clamped_coords[:,1].min():.6f}, {clamped_coords[:,1].max():.6f}]")
        print(f"  Clamped Z range: [{clamped_coords[:,2].min():.6f}, {clamped_coords[:,2].max():.6f}]")

with h5py.File(os.path.join(DATA_DIR, "thermal.h5"), "r") as f:
    t_grp = f["Function/Temperature"]
    t_keys = sorted(list(t_grp.keys()), key=lambda k: float(k.replace("_", ".")))
    print(f"\nThermal HDF5 step keys ({len(t_keys)} total):")
    print("  First 5:", t_keys[:5])
    print("  Last 5:", t_keys[-5:])

# 2. Inspect mechanical_results.jsonl structure (first and last lines without loading 943MB into RAM)
mech_jsonl_path = os.path.join(DATA_DIR, "mechanical_results.jsonl")
if os.path.exists(mech_jsonl_path):
    with open(mech_jsonl_path, "rb") as f:
        first_line = f.readline().decode("utf-8")
        # Seek near the end of the 943MB file to get the last line fast
        f.seek(max(0, os.path.getsize(mech_jsonl_path) - 5 * 1024 * 1024))
        tail_lines = f.read().decode("utf-8", errors="ignore").splitlines()
        last_line = tail_lines[-1] if tail_lines else ""

    first_rec = json.loads(first_line)
    last_rec = json.loads(last_line)
    print("\nInspecting mechanical_results.jsonl:")
    print("  Keys:", list(last_rec.keys()))
    print(f"  First time: {first_rec.get('time')} | Last time: {last_rec.get('time')}")
    for k, v in last_rec.items():
        if isinstance(v, list):
            elem_sample = v[0] if len(v) > 0 else None
            print(f"   * '{k}': len={len(v)}, sample={elem_sample}")

Geometry Bounding Box:
  X: min=0.000000, max=0.120000
  Y: min=0.000000, max=0.100000
  Z: min=0.000000, max=0.003000

Mechanical HDF5 step keys (51 total):
  First 5: ['0_02', '2', '4', '6', '8']
  Last 5: ['92', '94', '96', '98', '100']

Detected 4 clamped nodes (u=0 at t=100s):
  Clamped X range: [0.000000, 0.120000]
  Clamped Y range: [0.000000, 0.100000]
  Clamped Z range: [0.000000, 0.000000]

Thermal HDF5 step keys (501 total):
  First 5: ['0_02', '0_20000000000000001', '0_40000000000000002', '0_59999999999999998', '0_80000000000000004']
  Last 5: ['99_200000000000003', '99_400000000000006', '99_600000000000009', '99_799999999999997', '100']

Inspecting mechanical_results.jsonl:
  Keys: ['time', 'displacement_3d', 'equivalent_plastic_strain', 'temperatures', 'von_mises_stress', 'material_state']
  First time: 0.02 | Last time: 100.0
   * 'displacement_3d': len=16226, sample=[0.0, 0.0, 0.0]
   * 'equivalent_plastic_strain': len=13320, sample=0.02629901923966684
   * 'temperature

# Restart

In [9]:
!apt-get update
!apt-get install -y libglu1-mesa libopenmpi-dev

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:3 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease
Get:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:7 https://cli.github.com/packages stable/main amd64 Packages [358 B]
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu noble/main all Packages [10.6 MB]
Get:10 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:11 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [79.0 kB]
Get:12 http://security.ubuntu.com/ubuntu noble-security/multiverse amd64 Packages [55.6 kB]
Get:13 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 Packages [1,792 kB]
Get:14

In [10]:
# Step 4: Download and run the FEniCSx installation script.
!wget "https://fem-on-colab.github.io/releases/fenicsx-install-release-real.sh" -O "/tmp/fenicsx-install.sh" && bash "/tmp/fenicsx-install.sh"

--2026-11-15 03:32:14--  https://fem-on-colab.github.io/releases/fenicsx-install-release-real.sh
Resolving fem-on-colab.github.io (fem-on-colab.github.io)... 185.199.111.153, 185.199.109.153, 185.199.108.153, ...
Connecting to fem-on-colab.github.io (fem-on-colab.github.io)|185.199.111.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4339 (4.2K) [application/x-sh]
Saving to: ‘/tmp/fenicsx-install.sh’

/tmp/fenicsx-instal 100%[===================>]   4.24K  --.-KB/s    in 0s      

2026-11-15 03:32:14 (36.8 MB/s) - ‘/tmp/fenicsx-install.sh’ saved [4339/4339]

+ INSTALL_PREFIX=/usr/local
++ echo /usr/local
++ awk -F/ '{print NF-1}'
+ INSTALL_PREFIX_DEPTH=2
+ PROJECT_NAME=fem-on-colab
+ SHARE_PREFIX=/usr/local/share/fem-on-colab
+ FENICSX_INSTALLED=/usr/local/share/fem-on-colab/fenicsx.installed
+ [[ ! -f /usr/local/share/fem-on-colab/fenicsx.installed ]]
+ PYBIND11_INSTALL_SCRIPT_PATH=https://github.com/fem-on-colab/fem-on-colab.github.io/raw/c95043cb/release

In [11]:

!pip install mpi4py gmsh meshio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.0/40.0 MB 15.1 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [meshio]


In [13]:
import os
import json
import h5py
import numpy as np
import ufl
from dataclasses import dataclass
from scipy.spatial import cKDTree
from mpi4py import MPI
from petsc4py import PETSc
from dolfinx import fem, io
from dolfinx.fem.petsc import LinearProblem
from dolfinx.mesh import exterior_facet_indices, meshtags

# ============================================================
# 1. Exact Configuration from config.py (Extended 100s -> 110s)
# ============================================================
@dataclass
class RestartConfig:
    length: float = 0.120
    width: float = 0.10
    thickness: float = 0.003
    T0: float = 25.0
    Tamb: float = 25.0
    h_conv: float = 15.0
    emissivity: float = 0.70
    sigma_sb: float = 5.670374419e-8
    rho: float = 7850.0
    dt: float = 0.02
    output_every: int = 10
    t_start: float = 100.0
    t_end: float = 110.0

    plastic_inner_tol: float = 1.0e-6
    plastic_inner_max_it: int = 30
    plastic_relaxation: float = 0.7
    hardening_modulus: float = 1.0e9
    hardening_min_factor: float = 0.10
    hardening_T_ref: float = 400.0
    hardening_T_end: float = 1100.0
    T_anneal: float = 1000.0

cfg = RestartConfig()

DATA_DIR = "/content/welding_data_0"
OUT_DIR = os.path.join(DATA_DIR, "extension_100_to_110")
os.makedirs(OUT_DIR, exist_ok=True)

# ============================================================
# 2. Embedded Functions (material.py, postprocessor.py, mechanical.py)
# ============================================================
def _is_numeric(val):
    return isinstance(val, (int, float, list, tuple, np.ndarray, np.generic))

def conductivity(T, gum_factor=1.0):
    if not _is_numeric(T):
        k = ufl.conditional(ufl.lt(T, 800.0), 54.0 - 0.0333 * T, 27.3)
        k_eff = ufl.conditional(k >= 15.0, k, 15.0)
        return k_eff * gum_factor
    T = np.asarray(T, dtype=float)
    k = np.where(T < 800.0, 54.0 - 0.0333 * T, 27.3)
    return np.maximum(k, 15.0) * gum_factor

def heat_capacity(T):
    if not _is_numeric(T):
        return 650.0
    T = np.asarray(T, dtype=float)
    cp = np.full_like(T, 650.0)
    m1 = T < 600.0
    cp[m1] = 425.0 + 0.773 * T[m1] - 1.69e-3 * T[m1] ** 2 + 2.22e-6 * T[m1] ** 3
    m2 = (T >= 600.0) & (T < 735.0)
    cp[m2] = 666.0 + 13002.0 / np.maximum(738.0 - T[m2], 1.0)
    m3 = (T >= 735.0) & (T < 900.0)
    cp[m3] = 545.0 + 17820.0 / np.maximum(T[m3] - 731.0, 1.0)
    return np.clip(cp, 400.0, 5000.0)

def youngs_modulus(T, gum_factor=1.0):
    if not _is_numeric(T):
        E = ufl.conditional(ufl.lt(T, 100.0), 200e9, 200e9 * (1.0 - (T - 100.0) / 1000.0))
        E_eff = ufl.conditional(E >= 10e9, E, 10e9)
        return E_eff * gum_factor
    T = np.asarray(T, dtype=float)
    E = np.where(T < 100.0, 200e9, 200e9 * (1.0 - (T - 100.0) / 1000.0))
    return np.maximum(E, 10e9) * gum_factor

def poisson_ratio(T):
    if not _is_numeric(T):
        return 0.30
    return np.full_like(np.asarray(T, dtype=float), 0.30)

def yield_strength(T, gum_factor=1.0):
    if not _is_numeric(T):
        sy = ufl.conditional(ufl.lt(T, 400.0), 250e6, 250e6 * (1.0 - (T - 400.0) / 700.0))
        sy_eff = ufl.conditional(sy >= 5e6, sy, 5e6)
        return sy_eff * gum_factor
    T = np.asarray(T, dtype=float)
    sy = np.where(T < 400.0, 250e6, 250e6 * (1.0 - (T - 400.0) / 700.0))
    return np.maximum(sy, 5e6) * gum_factor

def expansion_coefficient(T):
    if not _is_numeric(T):
        return ufl.conditional(ufl.lt(T, 800.0), 1.4e-5, 1.6e-5)
    T = np.asarray(T, dtype=float)
    return np.where(T < 800.0, 1.4e-5, 1.6e-5)

def hardening_modulus(T, H0=1.0e9, H_min_factor=0.10, T_ref=400.0, T_end=1100.0, gum_factor=1.0):
    if not _is_numeric(T):
        factor = ufl.conditional(
            ufl.le(T, T_ref),
            1.0,
            ufl.conditional(
                ufl.ge(T, T_end),
                H_min_factor,
                1.0 - (1.0 - H_min_factor) * (T - T_ref) / (T_end - T_ref)
            )
        )
        return H0 * factor * gum_factor
    T = np.asarray(T, dtype=float)
    factor = np.ones_like(T)
    middle = (T > T_ref) & (T < T_end)
    factor[middle] = 1.0 - (1.0 - H_min_factor) * (T[middle] - T_ref) / (T_end - T_ref)
    factor[T >= T_end] = H_min_factor
    return H0 * factor * gum_factor

def eps(u):
    return ufl.sym(ufl.grad(u))

def dev(A):
    return A - ufl.tr(A) / 3.0 * ufl.Identity(3)

def get_interp_points(element):
    pts = element.interpolation_points
    return pts() if callable(pts) else pts

def create_mechanical_bcs(mesh, V, cfg):
    L = float(cfg.length)
    W = float(cfg.width)
    points = [
        np.array([0.0, 0.0, 0.0]),
        np.array([L,   0.0, 0.0]),
        np.array([0.0, W,   0.0]),
        np.array([L,   W,   0.0]),
    ]
    bcs = []
    for pnt in points:
        def marker(x, p=pnt):
            return np.logical_and.reduce(
                np.isclose(x.T, p, atol=1.0e-8),
                axis=1,
            )
        dofs = fem.locate_dofs_geometrical(V, marker)
        if len(dofs) > 0:
            zero = np.zeros(3, dtype=PETSc.ScalarType)
            bcs.append(fem.dirichletbc(zero, dofs, V))
    return bcs

def j2_radial_return(sigma_trial, sigma_y, mu, H):
    n_cells = sigma_trial.shape[0]
    identity = np.eye(3)
    p_trial = np.trace(sigma_trial, axis1=1, axis2=2) / 3.0
    s_trial = sigma_trial - p_trial[:, None, None] * identity[None, :, :]
    seq_trial = np.sqrt(
        np.maximum(0.0, 1.5 * np.sum(s_trial * s_trial, axis=(1, 2)))
    )

    d_eps_p = np.zeros_like(sigma_trial)
    d_eqp = np.zeros(n_cells, dtype=float)
    sigma_new = sigma_trial.copy()

    active = (seq_trial > sigma_y) & (seq_trial > 1.0e-14)
    active_ids = np.where(active)[0]

    if len(active_ids) > 0:
        delta_eqp = (seq_trial[active_ids] - sigma_y[active_ids]) / (
            3.0 * mu[active_ids] + H[active_ids]
        )
        delta_eqp = np.maximum(delta_eqp, 0.0)
        d_eqp[active_ids] = delta_eqp

        d_eps_p[active_ids] = (
            1.5
            * delta_eqp[:, None, None]
            * s_trial[active_ids]
            / seq_trial[active_ids, None, None]
        )

        seq_new = seq_trial[active_ids] - 3.0 * mu[active_ids] * delta_eqp
        seq_new = np.maximum(seq_new, 0.0)
        s_new_active = (seq_new / seq_trial[active_ids])[:, None, None] * s_trial[active_ids]
        sigma_new[active_ids] = (
            p_trial[active_ids, None, None] * identity[None, :, :] + s_new_active
        )

    return d_eps_p, d_eqp, sigma_new

def compute_stress_fields(mesh, displacement, temperature, plastic_strain, thermal_strain):
    DG0_scalar = fem.functionspace(mesh, ("DG", 0))
    DG0_tensor = fem.functionspace(mesh, ("DG", 0, (3, 3)))

    E = youngs_modulus(temperature)
    nu = poisson_ratio(temperature)
    mu = E / (2.0 * (1.0 + nu))
    lam = E * nu / ((1.0 + nu) * (1.0 - 2.0 * nu))

    eps_elastic = eps(displacement) - thermal_strain - plastic_strain
    sigma = lam * ufl.tr(eps_elastic) * ufl.Identity(3) + 2.0 * mu * eps_elastic

    sigma_func = fem.Function(DG0_tensor, name="StressTensor")
    sigma_func.interpolate(fem.Expression(sigma, get_interp_points(DG0_tensor.element)))
    sigma_func.x.scatter_forward()

    s = dev(sigma)
    vm_expr = ufl.sqrt(ufl.max_value(1.0e-24, 1.5 * ufl.inner(s, s)))
    vm_func = fem.Function(DG0_scalar, name="VonMisesStress")
    vm_func.interpolate(fem.Expression(vm_expr, get_interp_points(DG0_scalar.element)))
    vm_func.x.scatter_forward()

    return sigma_func, vm_func

# ============================================================
# 3. Load Mesh & Build Exact Index Mapping to HDF5 / JSONL
# ============================================================
mech_xdmf_path = os.path.join(DATA_DIR, "mechanical_elastoplastic.xdmf")
mech_h5_path = os.path.join(DATA_DIR, "mechanical_elastoplastic.h5")
therm_h5_path = os.path.join(DATA_DIR, "thermal.h5")

with io.XDMFFile(MPI.COMM_WORLD, mech_xdmf_path, "r") as xdmf:
    mesh = xdmf.read_mesh(name="Grid")

V = fem.functionspace(mesh, ("Lagrange", 1, (3,)))
Q = fem.functionspace(mesh, ("Lagrange", 1))
Qp_scalar = fem.functionspace(mesh, ("DG", 0))
Qp_tensor = fem.functionspace(mesh, ("DG", 0, (3, 3)))

with h5py.File(mech_h5_path, "r") as f:
    h5_geom = np.array(f["Mesh/Grid/geometry"], dtype=np.float64)
    h5_topo = np.array(f["Mesh/Grid/topology"], dtype=np.int64)
    u_100_h5 = np.array(f["Function/Displacement/100"], dtype=np.float64)
    eqp_100_h5 = np.array(f["Function/EquivalentPlasticStrain/100"], dtype=np.float64).flatten()
    state_100_h5 = np.array(f["Function/MaterialState/100"], dtype=np.float64).flatten()
    eps_p_100_h5 = np.array(f["Function/PlasticStrain/100"], dtype=np.float64)
    T_100_h5 = np.array(f["Function/Temperature/100"], dtype=np.float64).flatten()
    vm_100_h5 = np.array(f["Function/VonMisesStress/100"], dtype=np.float64).flatten()

dof_coords_nodal = Q.tabulate_dof_coordinates()
_, h5_node_for_dof = cKDTree(h5_geom).query(dof_coords_nodal)
dof_for_h5_node = np.argsort(h5_node_for_dof)

h5_cell_centroids = np.mean(h5_geom[h5_topo], axis=1)
dof_coords_cell = Qp_scalar.tabulate_dof_coordinates()
_, h5_cell_for_dof = cKDTree(h5_cell_centroids).query(dof_coords_cell)
dof_for_h5_cell = np.argsort(h5_cell_for_dof)

# ============================================================
# 4. Reconstruct Exact Cumulative Thermal Strain (eps_th) at t = 100s
# ============================================================
print("Reconstructing cumulative thermal strain (eps_th) from 0s to 100s...")
T_curr = fem.Function(Q, name="Temperature")
T_prev = fem.Function(Q, name="TemperaturePrevious")
eps_th = fem.Function(Qp_tensor, name="ThermalStrain")
d_eps_th = fem.Function(Qp_tensor)

T_prev.x.array[:] = cfg.T0
eps_th.x.array[:] = 0.0

dT_ufl = T_curr - T_prev
d_eps_th_expr = fem.Expression(
    expansion_coefficient(T_curr) * dT_ufl * ufl.Identity(3),
    get_interp_points(Qp_tensor.element)
)

with h5py.File(therm_h5_path, "r") as f:
    t_grp = f["Function/Temperature"]
    step_keys = sorted(list(t_grp.keys()), key=lambda k: float(k.replace("_", ".")))
    for k in step_keys:
        T_step_h5 = np.array(t_grp[k], dtype=np.float64).flatten()
        T_curr.x.array[:] = T_step_h5[h5_node_for_dof]
        T_curr.x.scatter_forward()
        d_eps_th.interpolate(d_eps_th_expr)
        eps_th.x.array[:] += d_eps_th.x.array
        T_prev.x.array[:] = T_curr.x.array

u = fem.Function(V, name="Displacement")
eps_p = fem.Function(Qp_tensor, name="PlasticStrain")
eqp = fem.Function(Qp_scalar, name="EquivalentPlasticStrain")
activated_mech = fem.Function(Qp_scalar, name="MaterialState")

u.x.array[:] = u_100_h5[h5_node_for_dof].flatten()
eps_p.x.array[:] = eps_p_100_h5[h5_cell_for_dof].flatten()
eqp.x.array[:] = eqp_100_h5[h5_cell_for_dof]
activated_mech.x.array[:] = state_100_h5[h5_cell_for_dof]

_, vm_check = compute_stress_fields(mesh, u, T_curr, eps_p, eps_th)
vm_diff = np.max(np.abs(vm_check.x.array - vm_100_h5[h5_cell_for_dof]))
print(f"State verification at t=100s -> Max Von Mises difference: {vm_diff:.4e} Pa")

# ============================================================
# 5. Phase 1: Thermal Restart Solve (100.0s -> 110.0s)
# ============================================================
print("\n--- Running Thermal Restart (100s -> 110s) ---")
T = fem.Function(Q, name="Temperature")
T_old = fem.Function(Q, name="Temperature")
T.x.array[:] = T_curr.x.array
T_old.x.array[:] = T_curr.x.array

tdim = mesh.topology.dim
fdim = tdim - 1
mesh.topology.create_connectivity(fdim, tdim)
facets = exterior_facet_indices(mesh.topology)
facet_tags = meshtags(mesh, fdim, facets, np.full(len(facets), 1, dtype=np.int32))

dx = ufl.Measure("dx", domain=mesh)
ds = ufl.Measure("ds", domain=mesh, subdomain_data=facet_tags)

v_t = ufl.TestFunction(Q)
u_t = ufl.TrialFunction(Q)
K = fem.Function(Qp_scalar, name="Conductivity")
CP = fem.Function(Qp_scalar, name="HeatCapacity")
T_dg_therm = fem.Function(Qp_scalar)
T_dg_therm_expr = fem.Expression(T_old, get_interp_points(Qp_scalar.element))

def update_thermal_props():
    T_dg_therm.interpolate(T_dg_therm_expr)
    T_cell = T_dg_therm.x.array
    K.x.array[:] = conductivity(T_cell)
    CP.x.array[:] = heat_capacity(T_cell)
    K.x.scatter_forward()
    CP.x.scatter_forward()

update_thermal_props()

rho_c = fem.Constant(mesh, PETSc.ScalarType(cfg.rho))
dt_c = fem.Constant(mesh, PETSc.ScalarType(cfg.dt))
T_amb_c = fem.Constant(mesh, PETSc.ScalarType(cfg.Tamb))
h_c = fem.Constant(mesh, PETSc.ScalarType(cfg.h_conv))
eps_c = fem.Constant(mesh, PETSc.ScalarType(cfg.emissivity))
sigma_c = fem.Constant(mesh, PETSc.ScalarType(cfg.sigma_sb))

Tprev_K = T_old + 273.15
Tamb_K = cfg.Tamb + 273.15
qrad = eps_c * sigma_c * (Tprev_K**4 - Tamb_K**4)
qconv = h_c * (T_old - T_amb_c)

a_therm = rho_c * CP / dt_c * u_t * v_t * dx + K * ufl.dot(ufl.grad(u_t), ufl.grad(v_t)) * dx
L_therm = rho_c * CP / dt_c * T_old * v_t * dx - (qconv + qrad) * v_t * ds(1)

therm_problem = LinearProblem(
    a_therm, L_therm, u=T,
    petsc_options_prefix="thermal_restart_",
    petsc_options={"ksp_type": "cg", "pc_type": "hypre", "pc_hypre_type": "boomeramg"}
)

therm_xdmf_out = io.XDMFFile(mesh.comm, os.path.join(OUT_DIR, "thermal_100_to_110.xdmf"), "w")
therm_xdmf_out.write_mesh(mesh)
therm_jsonl_file = open(os.path.join(OUT_DIR, "thermal_series_100_to_110.jsonl"), "w")

thermal_history_ext = []
n_substeps = int(round((cfg.t_end - cfg.t_start) / cfg.dt))

for step in range(1, n_substeps + 1):
    t = round(cfg.t_start + step * cfg.dt, 4)
    update_thermal_props()
    therm_problem.solve()
    T.x.scatter_forward()

    if step % cfg.output_every == 0:
        therm_xdmf_out.write_function(T, t)
        therm_xdmf_out.write_function(activated_mech, t)
        thermal_history_ext.append((t, T.x.array.copy()))

        therm_rec = {
            "time": t,
            "temperatures": T.x.array[dof_for_h5_node].tolist(),
            "material_state": activated_mech.x.array[dof_for_h5_cell].tolist()
        }
        therm_jsonl_file.write(json.dumps(therm_rec) + "\n")
        print(f"[thermal] t = {t:6.2f} s | T range: [{T.x.array.min():.2f}, {T.x.array.max():.2f}] C")

    T_old.x.array[:] = T.x.array
    T_old.x.scatter_forward()

therm_xdmf_out.close()
therm_jsonl_file.close()

# ============================================================
# 6. Phase 2: Mechanical Restart Solve (100.0s -> 110.0s)
# ============================================================
print("\n--- Running Mechanical Restart (100s -> 110s) ---")
v_m = ufl.TestFunction(V)
du_m = ufl.TrialFunction(V)

mu_func = fem.Function(Qp_scalar, name="Mu")
lam_func = fem.Function(Qp_scalar, name="Lambda")

def sigma_elastic(strain):
    return lam_func * ufl.tr(strain) * ufl.Identity(3) + 2.0 * mu_func * strain

a_mech = ufl.inner(sigma_elastic(eps(du_m)), eps(v_m)) * dx
bcs_mech = create_mechanical_bcs(mesh, V, cfg)

mech_xdmf_out = io.XDMFFile(mesh.comm, os.path.join(OUT_DIR, "mechanical_elastoplastic_100_to_110.xdmf"), "w")
mech_xdmf_out.write_mesh(mesh)
mech_jsonl_file = open(os.path.join(OUT_DIR, "mechanical_results_100_to_110.jsonl"), "w")

n_mech_steps = len(thermal_history_ext)

for step_idx, (t_val, Tarr) in enumerate(thermal_history_ext, start=1):
    T_curr.x.array[:] = Tarr
    T_curr.x.scatter_forward()

    T_dg = fem.Function(Qp_scalar)
    T_dg.interpolate(fem.Expression(T_curr, get_interp_points(Qp_scalar.element)))
    T_dg_np = T_dg.x.array

    E_np = youngs_modulus(T_dg_np)
    nu_np = poisson_ratio(T_dg_np)
    mu_func.x.array[:] = E_np / (2.0 * (1.0 + nu_np))
    lam_func.x.array[:] = E_np * nu_np / ((1.0 + nu_np) * (1.0 - 2.0 * nu_np))
    mu_func.x.scatter_forward()
    lam_func.x.scatter_forward()

    d_eps_th.interpolate(d_eps_th_expr)
    eps_th.x.array[:] += d_eps_th.x.array
    eps_th.x.scatter_forward()

    eps_p_start = eps_p.x.array.copy()
    eqp_start = eqp.x.array.copy()
    n_cells_local = eps_p.x.array.size // 9
    eps_p_start_3d = eps_p_start.reshape((n_cells_local, 3, 3))

    eigenstrain = eps_th + eps_p
    L_form = ufl.inner(sigma_elastic(eigenstrain), eps(v_m)) * dx

    problem = LinearProblem(
        a_mech, L_form, u=u, bcs=bcs_mech,
        petsc_options_prefix="mechanics_restart_",
        petsc_options={"ksp_type": "preonly", "pc_type": "lu", "pc_factor_mat_solver_type": "mumps"}
    )

    sy = fem.Function(Qp_scalar)
    sy.interpolate(fem.Expression(yield_strength(T_curr), get_interp_points(Qp_scalar.element)))
    mu_dg = fem.Function(Qp_scalar)
    mu_dg.interpolate(fem.Expression(youngs_modulus(T_curr) / (2.0 * (1.0 + poisson_ratio(T_curr))), get_interp_points(Qp_scalar.element)))
    H_dg = fem.Function(Qp_scalar)
    H_dg.interpolate(fem.Expression(
        hardening_modulus(T_curr, H0=cfg.hardening_modulus, H_min_factor=cfg.hardening_min_factor, T_ref=cfg.hardening_T_ref, T_end=cfg.hardening_T_end),
        get_interp_points(Qp_scalar.element)
    ))

    final_d_eps_p = None
    final_d_eqp = None

    for inner_it in range(cfg.plastic_inner_max_it):
        eps_p_guess = eps_p.x.array.copy()
        problem.solve()
        u.x.scatter_forward()

        eps_elastic_trial_expr = eps(u) - eps_th - eps_p
        sigma_trial = fem.Function(Qp_tensor)
        sigma_trial.interpolate(fem.Expression(sigma_elastic(eps_elastic_trial_expr), get_interp_points(Qp_tensor.element)))

        sigma_trial_np = sigma_trial.x.array.reshape((n_cells_local, 3, 3))
        sigma_y_np = sy.x.array + H_dg.x.array * eqp_start

        d_eps_p_step, d_eqp_step, _ = j2_radial_return(sigma_trial_np, sigma_y_np, mu_dg.x.array, H_dg.x.array)

        eps_p_target = eps_p_start_3d + d_eps_p_step
        eps_p_relaxed = (1.0 - cfg.plastic_relaxation) * eps_p_guess.reshape((n_cells_local, 3, 3)) + cfg.plastic_relaxation * eps_p_target
        eps_p.x.array[:] = eps_p_relaxed.reshape(-1)
        eps_p.x.scatter_forward()

        plastic_error = np.linalg.norm(eps_p.x.array - eps_p_guess) / max(np.linalg.norm(eps_p.x.array), 1.0e-16)
        final_d_eps_p = d_eps_p_step
        final_d_eqp = d_eqp_step

        if plastic_error < cfg.plastic_inner_tol:
            break

    eps_p.x.array[:] = (eps_p_start_3d + final_d_eps_p).reshape(-1)
    eqp.x.array[:] = eqp_start + final_d_eqp
    eps_p.x.scatter_forward()
    eqp.x.scatter_forward()

    problem.solve()
    u.x.scatter_forward()
    T_prev.x.array[:] = T_curr.x.array
    T_prev.x.scatter_forward()

    sigma_tensor, vm = compute_stress_fields(mesh, u, T_curr, eps_p, eps_th)

    if step_idx % cfg.output_every == 0 or step_idx == n_mech_steps:
        mech_xdmf_out.write_function(u, t_val)
        mech_xdmf_out.write_function(T_curr, t_val)
        mech_xdmf_out.write_function(eqp, t_val)
        mech_xdmf_out.write_function(eps_p, t_val)
        mech_xdmf_out.write_function(sigma_tensor, t_val)
        mech_xdmf_out.write_function(vm, t_val)
        mech_xdmf_out.write_function(activated_mech, t_val)

    mech_rec = {
        "time": t_val,
        "displacement_3d": u.x.array.reshape((-1, 3))[dof_for_h5_node].tolist(),
        "equivalent_plastic_strain": eqp.x.array[dof_for_h5_cell].tolist(),
        "temperatures": T_curr.x.array[dof_for_h5_node].tolist(),
        "von_mises_stress": vm.x.array[dof_for_h5_cell].tolist(),
        "material_state": activated_mech.x.array[dof_for_h5_cell].tolist()
    }
    mech_jsonl_file.write(json.dumps(mech_rec) + "\n")

    print(
        f"[mech {step_idx:02d}/{n_mech_steps}] t = {t_val:6.2f} s | "
        f"max |u| = {np.max(np.abs(u.x.array)):.5e} m | "
        f"max VM = {np.max(vm.x.array)/1e6:.2f} MPa"
    )

mech_xdmf_out.close()
mech_jsonl_file.close()
print(f"\nRestart completed successfully! Output files saved in: {OUT_DIR}")

Reconstructing cumulative thermal strain (eps_th) from 0s to 100s...
State verification at t=100s -> Max Von Mises difference: 0.0000e+00 Pa

--- Running Thermal Restart (100s -> 110s) ---
[thermal] t = 100.20 s | T range: [157.90, 187.83] C
[thermal] t = 100.40 s | T range: [157.84, 187.61] C
[thermal] t = 100.60 s | T range: [157.78, 187.39] C
[thermal] t = 100.80 s | T range: [157.72, 187.17] C
[thermal] t = 101.00 s | T range: [157.66, 186.95] C
[thermal] t = 101.20 s | T range: [157.60, 186.73] C
[thermal] t = 101.40 s | T range: [157.54, 186.52] C
[thermal] t = 101.60 s | T range: [157.47, 186.30] C
[thermal] t = 101.80 s | T range: [157.41, 186.09] C
[thermal] t = 102.00 s | T range: [157.35, 185.87] C
[thermal] t = 102.20 s | T range: [157.29, 185.66] C
[thermal] t = 102.40 s | T range: [157.23, 185.45] C
[thermal] t = 102.60 s | T range: [157.16, 185.24] C
[thermal] t = 102.80 s | T range: [157.10, 185.03] C
[thermal] t = 103.00 s | T range: [157.04, 184.82] C
[thermal] t = 10

# SAVE ON DRIVE

In [14]:
import shutil
import os
from google.colab import drive

drive.mount("/content/drive")

src_dir = "/content/welding_data_0/extension_100_to_110"
dst_dir = "/content/drive/MyDrive/welding_extension_100_to_110"

shutil.copytree(src_dir, dst_dir, dirs_exist_ok=True)
print("Successfully backed up to Google Drive:", os.listdir(dst_dir))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Successfully backed up to Google Drive: ['mechanical_elastoplastic_100_to_110.xdmf', 'mechanical_elastoplastic_100_to_110.h5', 'thermal_100_to_110.xdmf', 'mechanical_results_100_to_110.jsonl', 'thermal_series_100_to_110.jsonl', 'thermal_100_to_110.h5']


# Test save files

In [15]:
import os
import json
import numpy as np

DATA_DIR = "/content/welding_data_0"
OUT_DIR = os.path.join(DATA_DIR, "extension_100_to_110")

ext_mech_path = os.path.join(OUT_DIR, "mechanical_results_100_to_110.jsonl")
orig_mech_path = os.path.join(DATA_DIR, "mechanical_results.jsonl")

if not os.path.exists(ext_mech_path):
    print("ERROR: Output file not found! The simulation cell has not finished running.")
else:
    # Read last frame of original file (t = 100.0s)
    with open(orig_mech_path, "rb") as f:
        f.seek(max(0, os.path.getsize(orig_mech_path) - 5 * 1024 * 1024))
        last_orig_line = f.read().decode("utf-8", errors="ignore").splitlines()[-1]
    rec_100 = json.loads(last_orig_line)

    # Read all frames of the new extension file (100.2s -> 110.0s)
    with open(ext_mech_path, "r") as f:
        ext_lines = [line.strip() for line in f if line.strip()]

    print(f"Total new frames recorded: {len(ext_lines)} (Expected: 50)")
    rec_first = json.loads(ext_lines[0])
    rec_last = json.loads(ext_lines[-1])

    def summarize(label, r):
        T = np.array(r["temperatures"])
        u = np.linalg.norm(np.array(r["displacement_3d"]), axis=1)
        vm = np.array(r["von_mises_stress"]) / 1e6
        peeq = np.array(r["equivalent_plastic_strain"])
        print(
            f"{label} (t = {r['time']}s) -> "
            f"T: [{T.min():.2f}, {T.max():.2f}] C | "
            f"max |u|: {u.max()*1000:.4f} mm | "
            f"max VM: {vm.max():.2f} MPa | "
            f"max PEEQ: {peeq.max():.5f}"
        )

    summarize("Original End ", rec_100)
    summarize("Restart Start", rec_first)
    summarize("Restart End  ", rec_last)

Total new frames recorded: 50 (Expected: 50)
Original End  (t = 100.0s) -> T: [157.96, 188.06] C | max |u|: 1.3252 mm | max VM: 456.81 MPa | max PEEQ: 0.20176
Restart Start (t = 100.2s) -> T: [157.90, 187.83] C | max |u|: 1.3246 mm | max VM: 456.80 MPa | max PEEQ: 0.20179
Restart End   (t = 110.0s) -> T: [154.68, 178.03] C | max |u|: 1.2976 mm | max VM: 456.36 MPa | max PEEQ: 0.20277


In [16]:
import shutil
from google.colab import files

src_dir = "/content/welding_data_0/extension_100_to_110"
zip_path = "/content/extension_100_to_110"

shutil.make_archive(zip_path, "zip", src_dir)
files.download(zip_path + ".zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>